In [1]:
!pip install -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig
)
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np
# from huggingface_hub import interpreter_login

# interpreter_login()

In [3]:
import os
# disable Weights and Biases
os.environ['WANDB_DISABLED']="true"

In [4]:
from pynvml import *

def print_gpu_utilization():
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    print(f"GPU memory occupied: {info.used//1024**2} MB.")

# Load dataset

In [5]:
huggingface_dataset_name = "neil-code/dialogsum-test"
dataset = load_dataset(huggingface_dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

In [6]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

In [7]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

# Load base model

In [8]:
model_name='microsoft/phi-2'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True,
                                                      )

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,add_bos_token=True,
                                          use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [9]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

def gen(model,p, maxlen=100, sample=True):
    toks = eval_tokenizer(p, return_tensors="pt")
    res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,num_return_sequences=1,temperature=0.1,num_beams=1,top_p=0.95,).to('cpu')
    return eval_tokenizer.batch_decode(res,skip_special_tokens=True)

# Test model with zero-shot inference

In [10]:
%%time
from transformers import set_seed
seed = 42
set_seed(seed)

index = 10

prompt = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

formatted_prompt = f"Instruct: Summarize the following conversation.\n{prompt}\nOutput:\n"
res = gen(original_model,formatted_prompt,100,)
#print(res[0])
output = res[0].split('Output:\n')[1]

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{formatted_prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Instruct: Summarize the following conversation.
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
Output:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# attends Brian's birthday pa

# Preprocess dataset

In [11]:
def create_prompt_formats(sample):
    """
    Format various fields of the sample ('instruction','output')
    Then concatenate them using two newline characters
    :param sample: Sample dictionnary
    """
    INTRO_BLURB = "Below is an instruction that describes a task. Write a response that appropriately completes the request."
    INSTRUCTION_KEY = "### Instruct: Summarize the below conversation."
    RESPONSE_KEY = "### Output:"
    END_KEY = "### End"

    blurb = f"\n{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY}"
    input_context = f"{sample['dialogue']}" if sample["dialogue"] else None
    response = f"{RESPONSE_KEY}\n{sample['summary']}"
    end = f"{END_KEY}"

    parts = [part for part in [blurb, instruction, input_context, response, end] if part]

    formatted_prompt = "\n\n".join(parts)
    sample["text"] = formatted_prompt

    return sample

In [12]:
# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def get_max_length(model):
    conf = model.config
    max_length = None
    for length_setting in ["n_positions", "max_position_embeddings", "seq_length"]:
        max_length = getattr(model.config, length_setting, None)
        if max_length:
            print(f"Found max lenth: {max_length}")
            break
    if not max_length:
        max_length = 1024
        print(f"Using default max length: {max_length}")
    return max_length


def preprocess_batch(batch, tokenizer, max_length):
    """
    Tokenizing a batch
    """
    return tokenizer(
        batch["text"],
        max_length=max_length,
        truncation=True,
    )

In [13]:
from functools import partial

# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def preprocess_dataset(tokenizer: AutoTokenizer, max_length: int,seed, dataset):
    """Format & tokenize it so it is ready for training
    :param tokenizer (AutoTokenizer): Model Tokenizer
    :param max_length (int): Maximum number of tokens to emit from tokenizer
    """

    # Add prompt to each sample
    print("Preprocessing dataset...")
    dataset = dataset.map(create_prompt_formats)#, batched=True)

    _preprocessing_function = partial(preprocess_batch, max_length=max_length, tokenizer=tokenizer)
    dataset = dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=['id', 'topic', 'dialogue', 'summary'],
    )

    # Filter out samples that have input_ids exceeding max_length
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < max_length)

    # Shuffle dataset
    dataset = dataset.shuffle(seed=seed)

    return dataset

In [14]:
print_gpu_utilization()

GPU memory occupied: 2630 MB.


In [15]:
# ## Pre-process dataset
max_length = get_max_length(original_model)
print(max_length)

train_dataset = preprocess_dataset(tokenizer, max_length,seed, dataset['train'])
eval_dataset = preprocess_dataset(tokenizer, max_length,seed, dataset['validation'])

Found max lenth: 2048
2048
Preprocessing dataset...


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1999 [00:00<?, ? examples/s]

Preprocessing dataset...


Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Filter:   0%|          | 0/499 [00:00<?, ? examples/s]

In [16]:
print(f"Shapes of the datasets:")
print(f"Training: {train_dataset.shape}")
print(f"Validation: {eval_dataset.shape}")
print(train_dataset)

Shapes of the datasets:
Training: (1999, 3)
Validation: (499, 3)
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1999
})


In [17]:
train_dataset['text'][1]

"\nBelow is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruct: Summarize the below conversation.\n\n#Person1#: Let' s got out tomorrow night. We can go to a bar and try to find you a girlfriend. \n#Person2#: I don' t think that' s a good idea. I am just not good with approaching someone and starting up a conversation. \n#Person1#: Maybe you just need a few pick-up lines, you know, break the ice. \n#Person2#: Pick-up lines don' t work! \n#Person1#: Come on! You can just walk up to a girl and say'If you were a booger I' d pick you first. ' \n#Person2#: What? Come on! That's just lame! No girl would fall for that! \n#Person1#: Fine, then you can say, 'So there you are! I' ve been looking all over for YOU, the woman of my dreams! ' \n#Person2#: That' s a good one! I think that' s pretty funny. \n#Person1#: Yeah, so you make her laugh, you make a fool of yourself a little bit and then you buy her a drink. \n#Person2#: Ok, how do

In [18]:

def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    percentage = 100 * trainable_model_params / all_model_params
    summary = (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: {percentage:.4f}%"
    )
    return summary, trainable_model_params, all_model_params, percentage


In [19]:

# Experiment controls
base_model_id = "microsoft/phi-2"
SEED = 42
MAX_STEPS = 500          #changed from 1000 to 500
EVAL_SAMPLE_SIZE = 10
BASE_OUTPUT_DIR = "./phi2_lora_outputs"

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

DEFAULT_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'dense']
QV_ONLY_TARGET_MODULES = ['q_proj', 'v_proj']

EXPERIMENTS = {
    "default_lora": DEFAULT_TARGET_MODULES,
    "qv_only_lora": QV_ONLY_TARGET_MODULES,
}


In [20]:
import transformers

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import evaluate
import json

def load_quantized_base_model(model_id=base_model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,

    )
    return model

def build_peft_model(target_modules, model_id=base_model_id):
    model = load_quantized_base_model(model_id)
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    config = LoraConfig(
        r=32,
        lora_alpha=32,
        target_modules=target_modules,
        bias="none",
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )

    peft_model = get_peft_model(model, config)
    return peft_model, config

def build_trainer(model, output_dir):
    training_args = TrainingArguments(
        output_dir=output_dir,
        warmup_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        optim="paged_adamw_8bit",
        logging_steps=25,
        logging_dir=os.path.join(output_dir, "logs"),
        save_strategy="steps",
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        do_eval=True,
        gradient_checkpointing=True,
        report_to="none",
        seed=SEED,
    )

    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    return trainer, training_args

def generate_summary(model, dialogue, max_new_tokens=100):
    prompt = f"Instruct: Summarize the following conversation.\n{dialogue}\nOutput:\n"
    response = gen(model, prompt, max_new_tokens)[0]
    output = response.split("Output:\n", 1)[1] if "Output:\n" in response else response
    output = output.split("### End", 1)[0]
    output = output.split("#End", 1)[0]
    return output.strip()

def evaluate_model_rouge(model, dataset_obj, sample_size=EVAL_SAMPLE_SIZE):
    dialogues = dataset_obj['test'][0:sample_size]['dialogue']
    references = dataset_obj['test'][0:sample_size]['summary']

    predictions = []
    for dialogue in dialogues:
        predictions.append(generate_summary(model, dialogue))

    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_aggregator=True,
        use_stemmer=True,
    )
    return predictions, references, rouge_scores

def run_lora_experiment(experiment_name, target_modules):
    output_dir = os.path.join(BASE_OUTPUT_DIR, experiment_name)

    peft_model, lora_config = build_peft_model(target_modules)
    param_summary, trainable_params, all_params, trainable_pct = print_number_of_trainable_model_parameters(peft_model)
    print(param_summary)

    trainer, training_args = build_trainer(peft_model, output_dir)
    print(f"Training device: {training_args.device}")

    start_time = time.time()
    train_result = trainer.train()
    fine_tune_seconds = time.time() - start_time

    adapter_path = trainer.state.best_model_checkpoint or trainer.state.global_step
    trainer.save_model(output_dir)

    del trainer
    torch.cuda.empty_cache()

    base_model_for_eval = load_quantized_base_model()
    ft_model = PeftModel.from_pretrained(
        base_model_for_eval,
        output_dir,
        torch_dtype=torch.float16,
        is_trainable=False
    )

    predictions, references, rouge_scores = evaluate_model_rouge(ft_model, dataset, sample_size=EVAL_SAMPLE_SIZE)

    results = {
        "experiment_name": experiment_name,
        "target_modules": target_modules,
        "trainable_params": int(trainable_params),
        "all_params": int(all_params),
        "trainable_pct": float(trainable_pct),
        "fine_tune_seconds": float(fine_tune_seconds),
        "fine_tune_minutes": float(fine_tune_seconds / 60.0),
        "max_steps": int(MAX_STEPS),
        "eval_sample_size": int(EVAL_SAMPLE_SIZE),
        "rouge": rouge_scores,
        "output_dir": output_dir,
    }

    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return ft_model, results, predictions, references


In [21]:

# Sanity check: this is the vanilla quantized Phi-2 parameter count before LoRA
original_model_for_count = load_quantized_base_model()
base_param_summary, base_trainable, base_all, base_pct = print_number_of_trainable_model_parameters(original_model_for_count)
print(base_param_summary)
del original_model_for_count
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable model parameters: 263101440
all model parameters: 1521392640
percentage of trainable model parameters: 17.2935%


In [23]:
#load vanilla phi2 model
original_model = load_quantized_base_model()
original_model_predictions, original_references, original_model_results = evaluate_model_rouge(
    original_model,
    dataset,
    sample_size=EVAL_SAMPLE_SIZE
)

print("ORIGINAL MODEL:")
print(original_model_results)


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


ORIGINAL MODEL:
{'rouge1': np.float64(0.28840180279119754), 'rouge2': np.float64(0.10735049800388317), 'rougeL': np.float64(0.20864772812721782), 'rougeLsum': np.float64(0.21982071040830697)}


## Fine-tune experiment B: Q/V-only LoRA (`q_proj`, `v_proj`)

In [27]:
import time
start_time = time.time()

qv_ft_model, qv_results, qv_predictions, qv_references = run_lora_experiment(
    "qv_only_lora",
    QV_ONLY_TARGET_MODULES
)

print(json.dumps(qv_results, indent=2))


end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds") #record finetuning time



Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable model parameters: 10485760
all model parameters: 1531878400
percentage of trainable model parameters: 0.6845%
Training device: cuda:0


Step,Training Loss,Validation Loss
50,1.381919,1.368426
100,1.368721,1.351995
150,1.398064,1.342812
200,1.313925,1.338434
250,1.287937,1.334576
300,1.331861,1.332682
350,1.344051,1.330876
400,1.278503,1.330063
450,1.379460,1.329196
500,1.333736,1.328105


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{
  "experiment_name": "qv_only_lora",
  "target_modules": [
    "q_proj",
    "v_proj"
  ],
  "trainable_params": 10485760,
  "all_params": 1531878400,
  "trainable_pct": 0.6845034175036347,
  "fine_tune_seconds": 2700.2857761383057,
  "fine_tune_minutes": 45.004762935638425,
  "max_steps": 500,
  "eval_sample_size": 10,
  "rouge": {
    "rouge1": 0.3223341872031671,
    "rouge2": 0.10788650446342667,
    "rougeL": 0.24330614882772894,
    "rougeLsum": 0.2535055464987779
  },
  "output_dir": "./phi2_lora_outputs/qv_only_lora"
}
Total training time: 2795.87 seconds


## Quick qualitative example

In [28]:

from transformers import set_seed
set_seed(SEED)

index = 10
dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

print("-" * 100)
print("INPUT DIALOGUE:")
print(dialogue)
print("-" * 100)
print("HUMAN SUMMARY:")
print(summary)
print("-" * 100)
print("VANILLA PHI-2 SUMMARY:")
print(generate_summary(original_model, dialogue))
print("-" * 100)
print("Q/V-ONLY LORA SUMMARY:")
print(generate_summary(qv_ft_model, dialogue))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
----------------------------------------------------------------------------------------------------
HUMAN SUMMARY:
#Person1# attends Brian's birthday party. Brian thinks #Person1# looks great and charming.
--------

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Person1 and Person2 are at a party, and Person1 asks if they can have a dance. Person2 agrees and compliments Person1 on their appearance. Person1 thanks them and expresses their happiness with the party. Person2 agrees that it's a great party and suggests having a drink to celebrate.
----------------------------------------------------------------------------------------------------
Q/V-ONLY LORA SUMMARY:
Brian's birthday is being celebrated and #Person1# is invited to the party. #Person1# compliments Brian's appearance and they have a drink together to celebrate.


## Create side-by-side prediction table

In [29]:

import pandas as pd

comparison_df = pd.DataFrame({
    "reference_summary": original_references,
    "vanilla_phi2_summary": original_model_predictions,
    "qv_only_lora_summary": qv_predictions,
})

comparison_df

,reference_summary,vanilla_phi2_summary,qv_only_lora_summary
0,Ms. Dawson helps #Person1# to write a memo to ...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# asks Ms. Dawson to take a dictation ...
1,In order to prevent employees from wasting tim...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# asks Ms. Dawson to take a dictation ...
2,Ms. Dawson takes a dictation for #Person1# abo...,"Person 1: Ms. Dawson, I need you to take a dic...",#Person1# asks Ms. Dawson to take a dictation ...
3,#Person2# arrives late because of traffic jam....,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...
4,#Person2# decides to follow #Person1#'s sugges...,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...
5,#Person2# complains to #Person1# about the tra...,Person1 and Person2 are discussing the traffic...,#Person2# got stuck in traffic again and #Pers...
6,#Person1# tells Kate that Masha and Hero get d...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...
7,#Person1# tells Kate that Masha and Hero are g...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...
8,#Person1# and Kate talk about the divorce betw...,Kate informed that Masha and Hero are getting ...,Kate tells #Person1# that Masha and Hero are g...
9,#Person1# and Brian are at the birthday party ...,"Person1 and Person2 are at a party, and Person...",Brian's birthday party is a success. Brian's f...


In [30]:
#compute ROUGE score for vanilla and QV model
all_rouge_df = pd.DataFrame([
    {"model": "vanilla_phi2", **original_model_results},
    {"model": "qv_only_lora", **qv_results["rouge"]},
])

all_rouge_df

,model,rouge1,rouge2,rougeL,rougeLsum
0,vanilla_phi2,0.288402,0.107350,0.208648,0.219821
1,qv_only_lora,0.322334,0.107887,0.243306,0.253506


## Training time and trainable-parameter comparison

In [32]:

runtime_df = pd.DataFrame([
    {
        "model": "vanilla_phi2",
        "trainable_params": base_trainable,
        "trainable_pct": base_pct,
        "fine_tune_seconds": 0.0,
        "fine_tune_minutes": 0.0,
        "target_modules": "None (no fine-tuning)",
    },
    {
        "model": "qv_only_lora",
        "trainable_params": qv_results["trainable_params"],
        "trainable_pct": qv_results["trainable_pct"],
        "fine_tune_seconds": qv_results["fine_tune_seconds"],
        "fine_tune_minutes": qv_results["fine_tune_minutes"],
        "target_modules": ", ".join(qv_results["target_modules"]),
    },
])

runtime_df


,model,trainable_params,trainable_pct,fine_tune_seconds,fine_tune_minutes,target_modules
0,vanilla_phi2,263101440,17.293461,0.000000,0.000000,None (no fine-tuning)
1,qv_only_lora,10485760,0.684503,2700.285776,45.004763,"q_proj, v_proj"


In [33]:

all_rouge_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "rouge_scores.csv"), index=False)
runtime_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "runtime_and_params.csv"), index=False)
comparison_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "sample_predictions.csv"), index=False)

summary_report = {
    "baseline_rouge": original_model_results,
    "qv_only_lora": qv_results,
}

with open(os.path.join(BASE_OUTPUT_DIR, "assignment_q2_summary.json"), "w") as f:
    json.dump(summary_report, f, indent=2)

print(f"Saved outputs to: {BASE_OUTPUT_DIR}")


Saved outputs to: ./phi2_lora_outputs


In [34]:

print("Final ROUGE table")
display(all_rouge_df)

print("\nFinal runtime / parameter table")
display(runtime_df)


Final ROUGE table


,model,rouge1,rouge2,rougeL,rougeLsum
0,vanilla_phi2,0.288402,0.107350,0.208648,0.219821
1,qv_only_lora,0.322334,0.107887,0.243306,0.253506



Final runtime / parameter table


,model,trainable_params,trainable_pct,fine_tune_seconds,fine_tune_minutes,target_modules
0,vanilla_phi2,263101440,17.293461,0.000000,0.000000,None (no fine-tuning)
1,qv_only_lora,10485760,0.684503,2700.285776,45.004763,"q_proj, v_proj"


In [35]:

for _, row in runtime_df.iterrows():
    print(f"{row['model']}:")
    print(f"  trainable_params = {row['trainable_params']}")
    print(f"  trainable_pct = {row['trainable_pct']:.4f}%")
    print(f"  fine_tune_minutes = {row['fine_tune_minutes']:.2f}")
    print()


vanilla_phi2:
  trainable_params = 263101440
  trainable_pct = 17.2935%
  fine_tune_minutes = 0.00

qv_only_lora:
  trainable_params = 10485760
  trainable_pct = 0.6845%
  fine_tune_minutes = 45.00

